In [27]:
import json
import pandas as pd
import os

In [28]:
def extract_values(data, target_names):
    extracted_values = {}

    def traverse(obj):
        if isinstance(obj, dict):
            if 'name' in obj and obj['name'] in target_names and 'values' in obj:
                extracted_values[obj['name']] = obj['values']
            for key, value in obj.items():
                traverse(value)
        elif isinstance(obj, list):
            for item in obj:
                traverse(item)

    traverse(data)
    return extracted_values

# Function to extract only the 'values' from nested objects in specific columns
def clean_src_context_nested_values(dataframe):
    dataframe['src_context'] = dataframe['src_context'].apply(
        lambda x: x['values'] if isinstance(x, dict) and 'values' in x else x
    )
    return dataframe



# Function to extract only the 'values' field from lists of dictionaries
def extract_nested_values(column_data):
    values = column_data['values']
    print(values)
    new_values = []
    for val in values:
        new_values.extend(val['values'])
    return new_values

In [29]:
# Function to process a single JSON file
def process_json_file(file_path, target_names):
    with open(file_path, "r", encoding="utf-8") as file:
        json_data = json.load(file)

    # Extract values
    extracted_data = extract_values(json_data, target_names)

    # Convert the extracted data into a structured column-wise format
    max_length = max(len(v) for v in extracted_data.values()) if extracted_data else 0

    # Ensure all columns have the same number of rows by padding with None
    for key in extracted_data:
        while len(extracted_data[key]) < max_length:
            extracted_data[key].append(None)

    # Convert to DataFrame
    df_structured = pd.DataFrame(extracted_data)

    # Function to extract only the 'values' from nested objects in specific columns
    df_structured = clean_src_context_nested_values(df_structured)
    # Apply the function to the 'tgt_contexts' column
    if 'tgt_contexts' in df_structured.columns:
        df_structured['tgt_contexts'] = df_structured['tgt_contexts'].apply(extract_nested_values)


    return df_structured



In [30]:
# Function to process all JSON files in a directory
def process_all_json_in_directory(directory_path, target_names, output_csv_path):
    all_dataframes = []
    
    for filename in os.listdir(directory_path):
        if filename.endswith(".json"):
            file_path = os.path.join(directory_path, filename)
            print(f"Processing: {file_path}")
            df = process_json_file(file_path, target_names)
            df["source_file"] = filename  # Add a column to track the source file
            all_dataframes.append(df)

    # Combine all DataFrames
    final_df = pd.concat(all_dataframes, ignore_index=True)

    # Save to CSV
    final_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

    print(f"Saved combined DataFrame to {output_csv_path}")

In [31]:
# Function to process all JSON files in a directory
def process_single_json_file(directory_path, filename, target_names, output_csv_path):
    all_dataframes = []

    file_path = os.path.join(directory_path, filename)
    print(f"Processing: {file_path}")
    df = process_json_file(file_path, target_names)
    df["source_file"] = filename  # Add a column to track the source file
    all_dataframes.append(df)

    # Combine all DataFrames
    final_df = pd.concat(all_dataframes, ignore_index=True)

    # Save to CSV
    final_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

    print(f"Saved combined DataFrame to {output_csv_path}")

    return final_df

## for WIKIGAP data

In [36]:
# Target names to extract values from
# Directory containing JSON files
import urllib.parse
from datetime import datetime
import dill

today = datetime.today().date()
tgt_lang = "zh"
en_bio_id = "Wonton"
tgt_bio_id = "馄饨"
decoded_tgt_bio_id = urllib.parse.unquote(tgt_bio_id)
file_name = f"annotation_{today}_{en_bio_id}_{tgt_lang}.json"

json_directory = "/Users/anniewang/Desktop/infogap/scratch/ethics_annotation_save/wikigap_data"  # Change this to your directory path
output_csv = "wikigap_data_temp.csv"
target_names = {'fact', 'src_context', 'person_name', 'tgt_contexts', 'gpt-4o_intersection_label', 'language', 'paragraph_index'}

# # Process all JSON files in the directory and save the combined CSV
# process_all_json_in_directory(json_directory, target_names, output_csv)


df = process_single_json_file(json_directory, file_name, target_names, output_csv)

df

Processing: /Users/anniewang/Desktop/infogap/scratch/ethics_annotation_save/wikigap_data/annotation_2025-03-01_Wonton_zh.json
[{'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['Based on the Chinese method of making written characters, the radicals are changed from water to food.', "The characters then became 'hun tun' (餛飩, wonton in Cantonese)."]}, {'name': '', 'datatype': 'Utf8', 'bit_settings': 'SORTED_ASC', 'values': ['A wonton (traditional Chinese: 餛飩; simplified Chinese: 馄饨; pinyin: húntun; Jyutping: wan4 tan4) is a type of Chinese dumpling.']}]
[{'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['A related kind of wonton is made by using the same kind of wrapper but applying only a minute amount of filling.', 'The filling is frequently meat.']}, {'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['In Peruvian-Chinese gastronomic fusion called Chifa, wontons are called wantán in Peru.', 'In Chifa, wontons can be found fried with meat filling

,fact,person_name,src_context,tgt_contexts,paragraph_index,gpt-4o_intersection_label,language,source_file
0,「うどん」的汉字是「饂飩」,Wonton,"[雲吞在日本亦寫作「雲呑」。, 日本有一种面条叫「うどん」, 「うどん」的汉字是「饂飩」]",[Based on the Chinese method of making written...,17,no,zh,annotation_2025-03-01_Wonton_zh.json
1,馄饨馅料的种类包括咸鱼。,Wonton,"[馄饨馅料的种类包括香菇。, 馄饨馅料的种类包括香肠。, 馄饨馅料的种类包括咸鱼。]",[A related kind of wonton is made by using the...,20,yes,zh,annotation_2025-03-01_Wonton_zh.json
2,The Cantonese Yale romanization for wantan is ...,Wonton,[The traditional Chinese characters for wantan...,"[這種密封的包稱為渾沌。, 依據漢字造字的規則，後來才稱為餛飩。, 雲吞之名傳自廣東話。, ...",32,no,en,annotation_2025-03-01_Wonton_zh.json
3,这种面食称为“曲曲儿”。,Wonton,"[新疆菜中有一种以羊肉为馅的面食。, 这种面食类似馄饨。, 这种面食称为“曲曲儿”。]","[The wrapper-holding hand is quickly closed, s...",9,no,zh,annotation_2025-03-01_Wonton_zh.json
4,大馄饨一般用白开水煮熟。,Wonton,"[大馄饨外形呈大元宝形。, 大馄饨内馅主要有荠菜。, 大馄饨一般用白开水煮熟。]","[Big wontons are a large ingot shape., Big won...",11,yes,zh,annotation_2025-03-01_Wonton_zh.json
...,...,...,...,...,...,...,...,...
392,“曲曲儿”在维吾尔语中写作“چۆچۈرە”。,Wonton,"[这种面食类似馄饨。, 这种面食称为“曲曲儿”。, “曲曲儿”在维吾尔语中写作“چۆچۈرە”。]",[The Cantonese Yale romanization for wantan is...,9,no,zh,annotation_2025-03-01_Wonton_zh.json
393,元寶形餃子和南方大餛飩只是烹飪方式上有區別。,Wonton,"[在山東、河南、湖北、安徽等地流傳與南方大餛飩相似做法及形狀的元寶形餃子。, 元寶形餃子和南...","[Wontons are served in a variety of sizes., Th...",4,no,zh,annotation_2025-03-01_Wonton_zh.json
394,The two types of Ningbo wontons are steamed wo...,Wonton,"['Three delicacies wonton' contains pork, shri...","[元寶形餃子和南方大餛飩當同為宋人筆記裡的餶飿。, 元寶形餃子和南方大餛飩只是烹飪方式上有區...",19,no,en,annotation_2025-03-01_Wonton_zh.json
395,Jiaozi have more filling than wontons.,Wonton,[The triangular shape of wontons resembles a C...,[廣府雲吞傳入香港後，因為相比起河蝦，在香港更容易從海中捕獲海蝦，於是產生以海蝦取代河蝦的港...,5,no,en,annotation_2025-03-01_Wonton_zh.json


In [38]:
import loguru
logger = loguru.logger
BIO_SAVE_DIR = "/Users/anniewang/Desktop/infogap/scratch/wiki_food"
class BioFilenotFoundError(Exception):
    pass


def step_retrieve_prescraped_en_content_blocks(en_bio_id, 
                                              save_dir = BIO_SAVE_DIR,
                                               **kwargs):
    try:
        with open(f"{save_dir}/{en_bio_id}_en.pkl", 'rb') as f:
            return dill.load(f)
    except FileNotFoundError:
        raise BioFilenotFoundError(f"Could not find the prescraped bio file for {en_bio_id}")


def step_retrieve_prescraped_tgt_content_blocks(tgt_bio_id, tgt_lang: str, **kwargs):
    try: 
        logger.info(f"Retrieving {tgt_bio_id}_{tgt_lang} from {BIO_SAVE_DIR}")
        with open(f"{BIO_SAVE_DIR}/{tgt_bio_id}_{tgt_lang}.pkl", 'rb') as f:
            return dill.load(f)
    except FileNotFoundError:
        raise BioFilenotFoundError(f"Could not find the prescraped bio file for {tgt_bio_id}")


en_blocks = step_retrieve_prescraped_en_content_blocks(en_bio_id, save_dir = BIO_SAVE_DIR)
tgt_blocks = step_retrieve_prescraped_tgt_content_blocks(tgt_bio_id, tgt_lang, save_dir = BIO_SAVE_DIR)
   

2025-03-01 17:09:34.553 | INFO     | __main__:step_retrieve_prescraped_tgt_content_blocks:20 - Retrieving 馄饨_zh from /Users/anniewang/Desktop/infogap/scratch/wiki_food


In [44]:
en_blocks

[{'paragraph': 'A wonton (traditional Chinese: 餛飩; simplified Chinese: 馄饨; pinyin: húntun; Jyutping: wan4 tan4) is a type of Chinese dumpling commonly found across regional styles of Chinese cuisine. It is also spelled wantan or wuntun in transliteration from Cantonese 雲吞 / 云吞 (wan4 tan1) and wenden from Shanghainese 餛飩 / 馄饨 (hhun den). Even though there are many different styles of wonton served throughout China, Cantonese wontons are the most popular in the West due to the predominance of Cantonese restaurants overseas.'},
 {'paragraph': 'Wontons, which have their origins in China, have achieved significant popularity as a sought-after delicacy that is not only celebrated and enjoyed in East Asian cuisine, but also across various Southeast Asian culinary traditions.'},
 {'header': 'history'},
 {'paragraph': 'Yang Xiong from the western Han dynasty mentioned "bing wei zhi tun", which means wontons are a type of bread. The difference is that wontons have fillings inside and are eaten a

In [43]:
en_blocks_paragraph = [en_block['paragraph'] for en_block in en_blocks if 'paragraph' in en_block]
en_blocks_paragraph[32]

'In Cantonese, they are called wantan (simplified Chinese: 云吞; traditional Chinese: 雲吞; Jyutping: wan4 tan1; Cantonese Yale: wàhn tān), which means "cloud swallow" because when they are cooked, the dumplings float in the broth like small clouds.'

In [46]:
def process_paragraphs_with_headers(data):
    processed_data = []
    current_header = None
    paragraph_index = 0  # Start indexing paragraphs from 0

    for item in data:
        if "header" in item:
            current_header = item["header"]  # Update current header
        elif "paragraph" in item:
            # Append paragraph with its index and header reference
            processed_data.append({
                "index": paragraph_index,  # Assign paragraph index
                "paragraph": item["paragraph"],
                "parent_header": current_header  # Associate with last seen header
            })
            paragraph_index += 1  # Increment paragraph index

    return processed_data

processed_en_blocks = process_paragraphs_with_headers(en_blocks)

# Print output for verification
import json
print(json.dumps(processed_en_blocks,ensure_ascii=False))

[{"index": 0, "paragraph": "A wonton (traditional Chinese: 餛飩; simplified Chinese: 馄饨; pinyin: húntun; Jyutping: wan4 tan4) is a type of Chinese dumpling commonly found across regional styles of Chinese cuisine. It is also spelled wantan or wuntun in transliteration from Cantonese 雲吞 / 云吞 (wan4 tan1) and wenden from Shanghainese 餛飩 / 馄饨 (hhun den). Even though there are many different styles of wonton served throughout China, Cantonese wontons are the most popular in the West due to the predominance of Cantonese restaurants overseas.", "parent_header": null}, {"index": 1, "paragraph": "Wontons, which have their origins in China, have achieved significant popularity as a sought-after delicacy that is not only celebrated and enjoyed in East Asian cuisine, but also across various Southeast Asian culinary traditions.", "parent_header": null}, {"index": 2, "paragraph": "Yang Xiong from the western Han dynasty mentioned \"bing wei zhi tun\", which means wontons are a type of bread. The diffe

## for BENCHMARK project

In [26]:
# # Target names to extract values from
# # Directory containing JSON files
# json_directory = "/Users/anniewang/Desktop/infogap/scratch/ethics_annotation_save"  # Change this to your directory path
# output_csv = "combined_data.csv"
# target_names = {'fact', 'src_context', 'person_name', 'tgt_contexts', 'gpt-4o_intersection_label', 'sam_annotations', 'language', 'fact_index'}

# # # Process all JSON files in the directory and save the combined CSV
# # process_all_json_in_directory(json_directory, target_names, output_csv)

# file_name = "annotation_2025-02-28_Hummus.json"
# process_single_json_file(json_directory, file_name, target_names, output_csv)

Processing: /Users/anniewang/Desktop/infogap/scratch/ethics_annotation_save/annotation_2025-02-28_Hummus.json
[{'name': '', 'datatype': 'Utf8', 'bit_settings': 'SORTED_ASC', 'values': ['Hummus is served as an accompaniment to grilled chicken.']}, {'name': '', 'datatype': 'Utf8', 'bit_settings': 'SORTED_ASC', 'values': ['Hummus is served as an accompaniment to falafel.']}]
[{'name': '', 'datatype': 'Utf8', 'bit_settings': 'SORTED_ASC', 'values': ["A purée of chickpeas and tahini called hummus kasa appears in Muhammad bin Hasan al-Baghdadi's The Book of Dishes."]}, {'name': '', 'datatype': 'Utf8', 'bit_settings': 'SORTED_ASC', 'values': ['One survey found that 41% of Britons had hummus in their fridge, twice as many as the rest of Europe.']}]
[{'name': '', 'datatype': 'Utf8', 'bit_settings': 'SORTED_ASC', 'values': ["מסעדה המגישה מנות חומוס נקראת 'חומוסייה'."]}, {'name': '', 'datatype': 'Utf8', 'bit_settings': 'SORTED_ASC', 'values': ['לעיתים מנת חומוס בצלחת היא גם ארוחת בוקר']}]
[{'name

In [25]:
full = './wiki_text_process_test/english.csv'

full_df = pd.read_csv(full)

fact_index = [61, 82, 135, 117, 4, 143, 82, 69, 23, 99, 24, 24, 78, 111, 24]
intersection = []
for index in fact_index:
    intersection.extend(full_df[full_df['fact_index'] == index]['gpt-4o_intersection_label'])
    
intersection



['no',
 'no',
 'no',
 'no',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no']